In [ ]:
import Numeric.LinearAlgebra
import qualified Numeric.LinearAlgebra as LA ((<>))
import Numeric.LinearAlgebra.Data
import System.Random
import Data.Random.Normal

In [ ]:
nsample = 100
-- x = [ 10.0 * i / nsample | i <- [0..(nsample-1)] ]
x = linspace nsample (0, 10 :: Double)

In [ ]:
x2 = cmap (**2) x

In [ ]:
x0 = fromColumns [ fromList (replicate 100 1.0), x, x2 ]

In [ ]:
g <- getStdGen

In [ ]:
e = fromList $ take nsample $ normals g

In [ ]:
e

In [ ]:
beta = fromList [1, 0.1, 10]

In [ ]:
y = x0 #> beta + e

In [ ]:
y

In [ ]:
betaest = x0 <\>  y
betaest

In [ ]:
betavar = takeDiag $ inv (tr x0 LA.<> x0)
betavar

In [ ]:
sse = (y <# (ident 100 - x0 LA.<> inv (tr x0 LA.<> x0) LA.<> tr x0)) `dot` y
sse

In [ ]:
-- tval3 = 
sqrt (nsample - 3) * betaest ! 2 / sqrt ((betavar ! 2) * sse)

In [ ]:
-- tval3 = 
sqrt (nsample - 3) * (10.0103) / sqrt (betavar ! 2 * sse)

In [ ]:
spector <- loadMatrix "./spector_noheader.csv"
spector

# Reading a CSV

In [ ]:
import Data.Csv (FromRecord(..), ToRecord, HasHeader(..))
import GHC.Generics
import System.IO
import System.Exit (exitFailure)
-- from bytestring
import Data.ByteString (ByteString, hGetSome, empty)
import qualified Data.ByteString.Lazy as BL
import Data.Csv.Incremental (Parser(..), decode)

In [ ]:
import Data.Either (lefts, rights)

In [ ]:
import Numeric.LinearAlgebra ((<\>), takeDiag, inv, tr, dot, (<#), ident, (!))
import qualified Numeric.LinearAlgebra as LA ((<>))
import Numeric.LinearAlgebra.Data (fromLists, fromList, Vector, rows)

In [ ]:
data SpectorRecord = SpectorRecord
  { obs :: !Int
  , gpa  :: !Double
  , tuce :: !Int
  , psi :: !Int
  , grade :: !Int
  } deriving (Show, Eq, Generic)

instance FromRecord SpectorRecord
instance ToRecord SpectorRecord


In [ ]:
:info parseRecord

In [ ]:
feed :: (ByteString -> Parser SpectorRecord) -> Handle -> IO (Parser SpectorRecord)
feed k csvFile = do
  hIsEOF csvFile >>= \eof -> case eof of
    True  -> return $ k empty
    False -> k <$> hGetSome csvFile 4096

spector <- withFile "spector.csv" ReadMode $ \ csvFile -> do
  let loop !_ (Fail _ errMsg) = do putStrLn errMsg; return []
      loop acc (Many rs k)    = loop (acc <> rs) =<< feed k csvFile
      loop acc (Done rs)      = return $ (acc <> rs)
  loop [] (decode HasHeader)


In [ ]:
rights spector

In [ ]:
x0 = fromLists $ map (\spector -> [1.0, gpa spector, fromIntegral $ tuce spector, fromIntegral $ psi spector]) (rights spector)
y = fromList $ map (fromIntegral . grade) (rights spector) :: Vector Double

In [ ]:
x0

In [ ]:
y

In [ ]:
betaest = x0 <\>  y

In [ ]:
betavar = takeDiag $ inv (tr x0 LA.<> x0)
betavar

In [ ]:
sse = (y <# (ident (rows x0) - x0 LA.<> inv (tr x0 LA.<> x0) LA.<> tr x0)) `dot` y
sse

In [ ]:
-- const
sqrt (fromIntegral (rows x0) - 4) * betaest ! 0 / sqrt ((betavar ! 0) * sse)

In [ ]:
-- gpa
sqrt (fromIntegral (rows x0) - 4) * betaest ! 1 / sqrt ((betavar ! 1) * sse)

In [ ]:
-- tuce
sqrt (fromIntegral (rows x0) - 4) * betaest ! 2 / sqrt ((betavar ! 2) * sse)

In [ ]:
-- psi
sqrt (fromIntegral (rows x0) - 4) * betaest ! 3 / sqrt ((betavar ! 3) * sse)

```
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  GRADE   R-squared:                       0.416
Model:                            OLS   Adj. R-squared:                  0.353
Method:                 Least Squares   F-statistic:                     6.646
Date:                Thu, 14 Dec 2023   Prob (F-statistic):            0.00157
Time:                        14:55:37   Log-Likelihood:                -12.978
No. Observations:                  32   AIC:                             33.96
Df Residuals:                      28   BIC:                             39.82
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
GPA            0.4639      0.162      2.864      0.008       0.132       0.796
TUCE           0.0105      0.019      0.539      0.594      -0.029       0.050
PSI            0.3786      0.139      2.720      0.011       0.093       0.664
const         -1.4980      0.524     -2.859      0.008      -2.571      -0.425
==============================================================================
Omnibus:                        0.176   Durbin-Watson:                   2.346
Prob(Omnibus):                  0.916   Jarque-Bera (JB):                0.167
Skew:                           0.141   Prob(JB):                        0.920
Kurtosis:                       2.786   Cond. No.                         176.
==============================================================================
```
Taken from [statsmodels Spector linear regression example](https://www.statsmodels.org/stable/regression.html)

In [ ]:
import Statistics.Regression.Logistic (regress, Model(..))

In [ ]:
ys_ex :: [Double]
xs_ex :: [[Double]]
(ys_ex, xs_ex) = unzip $
  [ (1, [1, 1])
  , (0, [-1, -2])
  , (1, [2, 5])
  , (0, [-1, 1])
  , (1, [2, -1])
  , (1, [1, -10])
  , (0, [-0.1, 30])
  ]

In [ ]:
lr = 0.1 :: Double

coef0 :: [Double]
coef0 = [1, 0.1]

In [ ]:
res :: [Model [] Double]
res = regress lr ys_ex xs_ex coef0

In [ ]:
take 5 res

In [ ]:
res !! 10

In [ ]:
res !! 100

In [ ]:
res !! 1000

In [ ]:
res !! 10000

In [ ]:
res !! 100000

In [ ]:
res !! 150000

In [ ]:
res !! 200000

In [ ]:
res !! 500000

In [ ]:
res !! 1000000

In [ ]:
coefs = res !! 1000000

In [ ]:
logisticFunction :: Double -> Double
logisticFunction x = exp x / (1 + exp x)

In [ ]:
rawPredictions = map (logisticFunction . sum . zipWith (*) coefs) xs_ex
rawPredictions

In [ ]:
catPredictions = map (>= 0.5) rawPredictions
zip ys_ex catPredictions